In [ ]:

# Import truncation utility
try:
    from utils.tokenizer import truncate_text
    print("Truncation utility imported successfully.")
except ImportError:
    print("utils.tokenizer not found. Using placeholder truncation function.")
    def truncate_text(text, limit, **kwargs): return text
#install llama-cpp-python on colab w gpu support, optional for gguf models
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 50.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=33389726 sha256=d85b6553a42cc131ad3014f6e620dd34724a4e90516765e00f986ca06618a272
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import time

# Google AI Studio imports
try:
    from google import genai
    from google.colab import userdata
    print("Google AI Studio libraries imported.")
except ImportError:
    genai = None
    print("Google AI Studio libraries not found. Run `pip install -U google-generativeai` if needed.")

# google drive import
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator

# Detect device
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Try to import llama_cpp for GGUF support
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available.")
except ImportError:
    Llama = None
    print("llama-cpp-python is NOT installed. GGUF models will not be supported.")

Google AI Studio libraries imported.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator
Using device: cuda
llama-cpp-python is available.


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [ ]:
model_name = "unsloth/Qwen3-4B-Instruct-2507-GGUF"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False, n_ctx=32768) # Added n_ctx
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_0.gguf",
            verbose=False,
            n_gpu_layers=-1, # Offload to GPU if available
            n_ctx=80000 # Added n_ctx to utilize full context window
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Loading GGUF model: unsloth/Qwen3-4B-Instruct-2507-GGUF using llama-cpp-python...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_per_seq (80000) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


GGUF Model loaded successfully!


README Generation

In [ ]:
def generate_readme(parsed_file, model, tokenizer, token_limit=16384):
    # Check if the parsed file exists
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()

    # Truncate content to token limit
    code_content = truncate_text(code_content, token_limit, tokenizer=tokenizer)

    # Create a prompt
    prompt = f"Based on the following repository code structure and contents, generate a comprehensive README.md file. Include sections for: Description, Features, Installation, Usage, and any other relevant information.\n\n{code_content}\n\n"

    readme_content = ""

    # --- Inference Logic ---

    # 1. Google AI Studio (Gemini)
    # We check if 'client' is available in globals, which indicates the aistudio setup was run.
    if 'client' in globals() and client is not None and (model is None or hasattr(model, 'generate_content')):
        try:
            print(f"Generating content with Google AI Studio model: {real_model_name}...")
            # Using the updated client.models.generate_content syntax
            response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
            readme_content = response.text
        except Exception as e:
            print(f"Error generating content with Gemini: {e}")
            return ""

    # 2. llama-cpp-python
    elif model is not None and hasattr(model, 'create_completion'):
        print("Generating content with llama-cpp...")
        output = model.create_completion(
            prompt,
            max_tokens=4096,
            temperature=0.7,
            top_p=0.9,
            echo=False
        )
        readme_content = output['choices'][0]['text']

    # 3. Hugging Face Transformers
    elif model is not None and tokenizer is not None:
        print("Generating content with Transformers...")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=16384  # Adjusted for potentially large context
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4096,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)

    else:
        print("Error: No valid model configuration or client detected.")
        return ""

    # Post-processing
    if "README.md" in readme_content:
        # Attempt to split to get content after the header if the model echoes it
        parts = readme_content.split("README.md", 1)
        if len(parts) > 1:
            readme_content = parts[1].strip()

    # Clean up code blocks if the model wrapped the output
    if readme_content.startswith("```markdown"):
        readme_content = readme_content.replace("```markdown", "", 1)
    elif readme_content.startswith("```"):
        readme_content = readme_content.replace("```", "", 1)

    if readme_content.endswith("```"):
        readme_content = readme_content[:-3]

    return readme_content.strip()

In [ ]:
def save_readme(output_path, readme_content, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    filename = f"{prefix}generated_README.md" if prefix else "generated_README.md"
    output_path = os.path.join(output_path, filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_path}")
    return output_path

In [ ]:
import os
import time # Ensure time is imported, though it was in a previous cell

def process_all_parsed_files(model, tokenizer, parsed_files_dir, output_path):
    print("Starting README generation for all parsed files...")

    # Ensure output directory exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Iterate through all files in the parsed_files_dir
    for filename in os.listdir(parsed_files_dir):
        if filename.endswith("_parsed_code.txt"): # Assuming parsed files have this extension
            parsed_file_path = os.path.join(parsed_files_dir, filename)
            print(f"Processing: {parsed_file_path}")

            try:
                # Record start time
                time_start = time.time()

                # Generate the README
                readme_content = generate_readme(parsed_file_path, model, tokenizer)

                # Record end time
                time_end = time.time()
                time_elapsed = time_end - time_start

                if readme_content:
                    # Extract a prefix for the output filename (e.g., from 'apache_cordova-plugin-splashscreen_parsed_code.txt' to 'apache_cordova-plugin-splashscreen_')
                    prefix = filename.replace('_parsed_code.txt', '_')

                    # Print the generated README content and time taken
                    print("")
                    print(f"Generated README for {filename}:")
                    print(readme_content)
                    print(f"Time taken for {filename}: {time_elapsed:.2f} seconds")
                    print("")

                    # Save the generated README
                    save_readme(output_path, readme_content, prefix=prefix)
                else:
                    print(f"Skipping README generation for {filename} due to empty content.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
    print("Finished README generation for all parsed files.")

In [ ]:
parsed_files_dir = "./out"
output_path = "llm_output"

process_all_parsed_files(model, tokenizer, parsed_files_dir, output_path)

Starting README generation for all parsed files...
Processing: ./out/apache_cordova-plugin-splashscreen_parsed_code.txt
Generating content with llama-cpp...

Generated README for apache_cordova-plugin-splashscreen_parsed_code.txt:
Size: 7220 bytes
Lines: 182
---
---
title: Browser Splashscreen
description: Control the browser platform splash screen for your app.
---
<!--
# license: Licensed to the Apache Software Foundation (ASF) under one
#         or more contributor license agreements.  See the NOTICE file
#         distributed with this work for additional information
#         regarding copyright ownership.  The ASF licenses this file
#         to you under the Apache License, Version 2.0 (the
#         "License"); you may not use this file except in compliance
#         with the License.  You may obtain a copy of the License at
#
#           http://www.apache.org/licenses/LICENSE-2.0
#
#         Unless required by applicable law or agreed to in writing,
#         software distrib

In [41]:
#install llama-cpp-python on colab w gpu support, optional for gguf models
#!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

#Run git clone on Google Colab
!git clone https://github.com/SmallChungus1/ReadmeGenerator.git

# Installs rouge_score for comparasion
#!pip install evaluate rouge_score

fatal: destination path 'ReadmeGenerator' already exists and is not an empty directory.


Importing Google AI Studio

In [42]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import evaluate
import re
import pandas as pd

# Google AI Studio imports
try:
    from google import genai
    from google.colab import userdata
    print("Google AI Studio libraries imported.")
except ImportError:
    genai = None
    print("Google AI Studio libraries not found. Run `pip install -U google-generativeai` if needed.")

# Detect device
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Try to import llama_cpp for GGUF support
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available.")
except ImportError:
    Llama = None
    print("llama-cpp-python is NOT installed. GGUF models will not be supported.")

Google AI Studio libraries imported.
Using device: cpu
llama-cpp-python is NOT installed. GGUF models will not be supported.


In [43]:
model_name = "aistudio-gemini-2.5-flash"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False)
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_K_M.gguf",
            verbose=False,
            n_gpu_layers=-1 # Offload to GPU if available
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Configuring Google AI Studio model: gemini-2.5-flash...
Google AI Studio Model configured successfully!
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-p

Saving the README in llm_output directory

In [45]:
def save_readme(output_path, readme_content, parsed_file, prefix=""):
    # Ensure the directory exists before saving
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Created directory: {output_path}")

    # Saves the generated README.md content to a file in the specified output path
    original_filename = os.path.basename(parsed_file)
    base_name = os.path.splitext(original_filename)[0]
    filename = f"{prefix}{base_name}_README.md" if prefix else f"{base_name}_README.md"
    output_file = os.path.join(output_path, filename)

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_file}")
    return output_file

Generating the README from a repository

In [46]:
parsed_file = "/content/ReadmeGenerator/out/apache_cordova-plugin-splashscreen_parsed_code.txt"
output_path = "/content/ReadmeGenerator/llm_output"

readme = generate_readme(parsed_file, model, tokenizer, token_limit=16384)
if readme:
    print("README generated successfully.")
else:
    print("Failed to generate README.")

Generating content with Google AI Studio model: gemini-2.5-flash...
README generated successfully.


In [47]:
save_readme(output_path, readme, parsed_file)

README.md has been saved to /content/ReadmeGenerator/llm_output/apache_cordova-plugin-splashscreen_parsed_code_README.md


'/content/ReadmeGenerator/llm_output/apache_cordova-plugin-splashscreen_parsed_code_README.md'

Baseline vs. Ground Truth Setup

In [48]:
actual_readme_path = '/content/ReadmeGenerator/out/apache_cordova-plugin-splashscreen_README.md'
generated_readme_path = '/content/ReadmeGenerator/llm_output/apache_cordova-plugin-splashscreen_parsed_code_README.md'

with open(actual_readme_path, 'r', encoding='utf-8') as f:
    actual_readme_content = f.read()

with open(generated_readme_path, 'r', encoding='utf-8') as f:
    generated_readme_content = f.read()

Generates the Baseline vs. Ground Truth Metrics

In [49]:
def evaluate_readmes(reference_content, generated_content, client):
    prompt = f"""Please evaluate and compare the following two versions of a README.md file for the same repository.

    ### Reference README Content:
    {reference_content}

    ### Generated README Content:
    {generated_content}

    ### Evaluation Criteria:
    1. **Content Accuracy**: How well does the generated version match the technical facts and source details in the reference?
    2. **Formatting Quality**: Evaluate the markdown structure, header hierarchy, and overall readability.
    3. **Technical Completeness**: Check the coverage of essential sections like Description, Features, Installation, and Usage.

    ### Instructions:
    - Provide a score (1-10) for each of the three metrics above.
    - Provide a final summary of findings highlighting strengths and weaknesses of the generated version compared to the reference.
    """

    try:
        print(f"Sending evaluation prompt to model: {real_model_name}...")
        response = client.models.generate_content(
            model=f"models/{real_model_name}",
            contents=prompt
        )
        return response.text
    except Exception as e:
        print(f"Error during evaluation: {e}")
        return None

print("Function 'evaluate_readmes' defined successfully.")

Function 'evaluate_readmes' defined successfully.


Evaluating the Two READMEs

In [50]:
# Call the evaluation function
evaluation_results = evaluate_readmes(actual_readme_content, generated_readme_content, client)

# Check and display the results
if evaluation_results:
    print("Evaluation Results:\n")
    print(evaluation_results)
else:
    print("Failed to retrieve evaluation results.")

Sending evaluation prompt to model: gemini-2.5-flash...
Evaluation Results:

Here's an evaluation and comparison of the two README versions:

### Evaluation Scores:

1.  **Content Accuracy**: 10/10
2.  **Formatting Quality**: 9/10
3.  **Technical Completeness**: 10/10

---

### Final Summary of Findings:

The **Generated README Content** is a significant and substantial improvement over the **Reference README Content**. It takes the core information from the reference and expands upon it with better structure, clearer explanations, and additional valuable context, resulting in a much more professional and user-friendly document.

**Strengths of the Generated Version:**

1.  **Enhanced Technical Completeness and Clarity**:
    *   **New Sections**: The generated version introduces critical sections like "Description," "Features," "Usage," "Contributing," and "License." These are standard for any well-maintained repository and provide a much richer overview of the plugin.
    *   **Detai

Loading the rouge score through comparing generated and actual README

In [51]:
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=[generated_readme_content], references=[actual_readme_content])
rouge_l_score = results['rougeL']

print(f"ROUGE-L Score: {rouge_l_score}")

ROUGE-L Score: 0.5278875713658322


Calculating Perplexity Score

In [52]:
def calculate_perplexity(text, model_id='gpt2'):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

    encodings = tokenizer(text, return_tensors='pt')
    max_length = model.config.n_positions
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

perplexity_score = calculate_perplexity(generated_readme_content)
print(f"Perplexity Score: {perplexity_score}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (3187 > 1024). Running this sequence through the model will result in indexing errors


Perplexity Score: 8.766913414001465


Grading the Two READMEs with a Rubric

In [53]:
def grade_with_rubric(actual_readme, generated_readme, client):
    prompt = f"""You are a technical documentation expert. Please grade the following generated README based on its alignment with the provided source code structure.
    \n\n### Rubric (Scale 1-5):\n- 1 (Poor): Inaccurate, missing major technical details, or hallucinates features not in the source.
    \n- 2 (Fair): Significant omissions or several technical inaccuracies.
    \n- 3 (Good): Generally accurate and covers most major features, but lacks depth or has minor errors.
    \n- 4 (Very Good): Accurate, comprehensive, and well-aligned with the source code.
    \n- 5 (Excellent): Perfectly aligned with source code, highly comprehensive, and technically precise.\n\n###
    Source Code Context:\n{actual_readme}\n\n### Generated README Content:\n{generated_readme}\n\n###
    Instructions:\nProvide a numeric score (1-5) and a detailed justification for the grade."""

    try:
        print(f"Evaluating README with rubric using model: {real_model_name}...")
        response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
        return response.text
    except Exception as e:
        print(f"Error during rubric grading: {e}")
        return None

# Runs Rubric Evaluation
rubric_evaluation = grade_with_rubric(actual_readme_content, generated_readme_content, client)

if rubric_evaluation:
    print("\n--- Rubric Evaluation Results ---\n")
    print(rubric_evaluation)

    score_search = re.search(r'Grade:\s*(\d)', rubric_evaluation)
    if score_search:
        rubric_score = int(score_search.group(1))
    else:
        rubric_score = "N/A"
    print(f"\nExtracted Rubric Score: {rubric_score}")


Evaluating README with rubric using model: gemini-2.5-flash...

--- Rubric Evaluation Results ---

**Grade: 3 (Good)**

**Justification:**

The generated README is generally accurate in extracting and explaining the core functionalities, preferences, and methods of the `cordova-plugin-splashscreen` as presented in the source code context. It successfully identifies and details the `Installation`, `Methods` (`hide`, `show`), and `Preferences` sections (including `AutoHideSplashScreen`, `SplashScreenDelay`, `FadeSplashScreen`, `FadeSplashScreenDuration`, and the browser-specific quirks). The explanation of the `SplashScreenDelay`'s deprecation and `FadeSplashScreenDuration`'s interaction with `SplashScreenDelay` is also well-reproduced.

However, the README falls short on "alignment with the provided source code structure" and introduces several elements not explicitly present in the *provided source text*.

**Strengths:**
*   **Technical Accuracy:** Most technical details regarding pref

Judging if Generated README is Superior than Actual README

In [54]:
def run_win_rate_analysis(generated, reference, client):
    prompt = f"""You are a judge comparing two versions of a README.md file for the same software project.
    \n\n### Version A (Generated):\n{generated}\n\n### Version B (Ground Truth/Reference):\n{reference}\n\n### Task:\n
    Which version is better for a developer using this library? Consider clarity, completeness, and modern documentation standards.\n\n###
    Instructions:\n1. Provide a brief comparison.\n2. State clearly which one wins: 'Winner: Version A' or 'Winner: Version B'."""

    try:
        print(f"Performing Win Rate analysis using model: {real_model_name}...")
        response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
        return response.text
    except Exception as e:
        print(f"Error during win rate analysis: {e}")
        return None

# Runs Win Rate Analysis
win_rate_result = run_win_rate_analysis(generated_readme_content, actual_readme_content, client)

if win_rate_result:
    print("\n--- Win Rate Analysis Result ---\n")
    print(win_rate_result)
    win_status = "Win" if "Winner: Version A" in win_rate_result else "Loss"
    print(f"\nOutcome for Generated README: {win_status}")
else:
    win_status = "N/A"

Performing Win Rate analysis using model: gemini-2.5-flash...

--- Win Rate Analysis Result ---

### Comparison

**Version A (Generated)** is a comprehensive and well-structured README that excels in clarity, completeness, and adherence to modern documentation standards. It starts with a clear description, features list, and an explicit statement about supported platforms, including crucial historical context for platform support. The usage and configuration sections are meticulously detailed, providing clear explanations for each preference, including nuanced notes about `SplashScreenDelay` and `FadeSplashScreenDuration`. A standout feature is the dedicated "Splash Screen Image Configuration" section, which addresses a common pain point with clear examples and directory structure. It also includes important "Contributing" and "License" sections.

**Version B (Ground Truth/Reference)** is functional but less complete and less organized than Version A. Its initial description is brief, 

Overall Score Breakdown

In [55]:
report_data = {
    "Metric": ["ROUGE-L Score", "Perplexity", "Rubric Score (1-5)", "Win Rate Analysis Outcome"],
    "Value": [
        f"{rouge_l_score:.4f}",
        f"{perplexity_score:.4f}",
        f"{rubric_score}",
        f"{win_status}"
    ],
    "Interpretation": [
        "Measures overlap with ground truth (Higher is better)",
        "Measures linguistic fluency (Lower is better)",
        "Expert judge alignment with source code",
        "Comparison against ground truth baseline"
    ]
}

# Create a DataFrame for better visualization
evaluation_df = pd.DataFrame(report_data)

print("=== Consolidated README Evaluation Report ===\n")
print(evaluation_df.to_string(index=False))

print("\n--- Detailed Summary ---")
print(f"The generated README achieved a ROUGE-L score of {rouge_l_score:.4f} and a Perplexity of {perplexity_score:.4f}.")
print(f"The LLM Judge assigned a Rubric Score of {rubric_score}/5, highlighting high technical accuracy to current code.")
print(f"In the head-to-head battle, the outcome was a '{win_status}' due to specific functional omissions noted by the judge.")

=== Consolidated README Evaluation Report ===

                   Metric  Value                                        Interpretation
            ROUGE-L Score 0.5279 Measures overlap with ground truth (Higher is better)
               Perplexity 8.7669         Measures linguistic fluency (Lower is better)
       Rubric Score (1-5)      3               Expert judge alignment with source code
Win Rate Analysis Outcome    Win              Comparison against ground truth baseline

--- Detailed Summary ---
The generated README achieved a ROUGE-L score of 0.5279 and a Perplexity of 8.7669.
The LLM Judge assigned a Rubric Score of 3/5, highlighting high technical accuracy to current code.
In the head-to-head battle, the outcome was a 'Win' due to specific functional omissions noted by the judge.
